# Phonon dispersion (DFT) — siman + phonopy

Finite-displacement phonon bands for the lowest-Gibbs DFT cells:

- **layered P2₁/c** — already a 2×2×2 supercell (`layered_P2_1_c.sc`, `phonons_setup_eos` v100)
- **diaspore α** (Pnma) — force-relaxed 4×3×1 supercell (`diaspore_alpha.su.ifn`, `phonons_setup_rel` v100)

Every lattice vector is longer than 10 Å, so **DIM = 1 1 1** (no extra supercell). Displacements use **0.0075 Å**; `primitive_matrix='auto'` keeps band paths in the primitive BZ.

1. Write the relaxed `.end` structures from the siman database
2. `phonopy` displacements → one zen4 job **per displacement** (`phonons_disp`: `EDIFF=1e-8`, `LREAL=.FALSE.`)
3. Optional **NAC**: DFPT `LEPSILON` on the phonopy primitive → `BORN` → **LO–TO splitting**
4. Collect finished OUTCARs, build `FORCE_SETS`, plot bands (NAC applied if `BORN` exists)

Flags: `LAUNCH_JOBS` / `LAUNCH_NAC` submit; `READ_AND_PROCESS` reads finished jobs and runs phonopy.


In [1]:
%%capture
import os
import glob
import shutil
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import siman
from siman.calc_manage import smart_structure_read, add, res
from siman import header
from siman.database import write_database, read_database
from siman.set_functions import read_vasp_sets
from siman.header import db, _update_configuration
from pydoc import importfile

_update_configuration("project_conf.py")
read_database()
project_sets = importfile("project_sets.py")
varset = read_vasp_sets(project_sets.user_vasp_sets, override_global=1)

header.PATH2EDITOR = "notepad.exe"
header.PATH2POTENTIALS = "/home/a.burov/soft/vasp_potentials/potpaw_PBE_MPIE/"


In [2]:
from ase.io import read, write
from ase import Atoms
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import write_FORCE_SETS, parse_FORCE_SETS
from phonopy.interface.vasp import parse_set_of_forces, write_vasp

try:
    from phonopy import load as phonopy_load
except ImportError:
    try:
        from phonopy.cui.load import load as phonopy_load
    except ImportError:
        phonopy_load = None


In [3]:
# Force-relaxed α used for phonons:
# db["diaspore_alpha.su.ifn", "phonons_setup_rel", 100]
# db["diaspore_alpha.su", "phonons_setup_eos", 100].run("phonons_setup_rel", run=2, add=2, cluster="razor128_zen")
# db["diaspore_alpha.su.ifn", "phonons_setup_rel", 100].res(cluster="razor128")


In [4]:
# db["diaspore_alpha.su.ifn", "phonons_setup_rel", 100].res()


In [5]:
CLUSTER_ROOT = Path("/home/a.burov/icys_2025/niohf")
LOCAL_ROOT = Path("/home/arseniy/Desktop/work/niohf")
ROOT = CLUSTER_ROOT if CLUSTER_ROOT.exists() else LOCAL_ROOT

DISP_ROOT = ROOT / "phonons_dispersion"
FIGURES_ROOT = LOCAL_ROOT / "figures" / "dispersion"
if not LOCAL_ROOT.exists():
    FIGURES_ROOT = ROOT / "figures" / "dispersion"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

VASP_SET = "phonons_disp"       # forces: EDIFF=1e-8, LREAL=.FALSE.
NAC_SET = "phonons_nac"         # LEPSILON DFPT → Born + ε∞
CLUSTER = "zen4"
CALC_VER = 1

# Displacement amplitude (Å). Midway between 0.005 and 0.01 for H-containing cells.
DISP_DISTANCE = 0.0075

# True: phonopy -d + siman add() for force jobs
LAUNCH_JOBS = True
# True: submit DFPT NAC (Born / dielectric) on phonopy primitive
LAUNCH_NAC = False
# True: res() finished jobs and run phonopy when the set is complete
READ_AND_PROCESS = False
# Apply NAC / LO–TO when BORN exists in the phase folder
USE_NAC = True
SKIP_EXISTING = True

print(f"ROOT = {ROOT}")
print(f"DISP_ROOT = {DISP_ROOT}")
print(f"FIGURES_ROOT = {FIGURES_ROOT}")
print(f"set = {VASP_SET}  nac = {NAC_SET}  cluster = {CLUSTER}")
print(f"DISP_DISTANCE = {DISP_DISTANCE} Å")


ROOT = /home/a.burov/icys_2025/niohf
DISP_ROOT = /home/a.burov/icys_2025/niohf/phonons_dispersion
FIGURES_ROOT = /home/a.burov/icys_2025/niohf/figures/dispersion
set = phonons_disp  nac = phonons_nac  cluster = zen4
DISP_DISTANCE = 0.0075 Å


In [4]:
# Already-relaxed supercells (lattice vectors all > 10 A) → DIM = 1 1 1.
# Band paths are for the primitive cell (phonopy primitive_matrix='auto').
# layered P21/c: EOS volume (phonons_setup_eos).
# diaspore alpha: ionic force relax of EOS cell (phonons_setup_rel).
# layered P21/c: Z=(0,1/2,0) is the unique-b / stacking-related direction.
# diaspore alpha (Pnma): Z=(0,0,1/2) is along the long c axis.

IDENTITY = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]

PHASES = {
    "layered_P2_1_c": {
        "db_key": ("layered_P2_1_c.sc", "phonons_setup_eos", 100),
        "structure": ROOT / "optimized_best/dft/layered_P2_1_c.vasp",
        "dim": IDENTITY,
        "spacegroup": "P2_1/c",
        "paths": {
            "out_of_plane": {
                "labels": [r"$\Gamma$", "Z", "D", "B", r"$\Gamma$"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [0.0, 0.5, 0.0],
                    [0.0, 0.5, 0.5],
                    [0.0, 0.0, 0.5],
                    [0.0, 0.0, 0.0],
                ],
            },
            "in_plane": {
                "labels": [r"$\Gamma$", "A", "E", "Z", r"C$_2$", r"Y$_2$", r"$\Gamma$"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [-0.5, 0.0, 0.5],
                    [-0.5, 0.5, 0.5],
                    [0.0, 0.5, 0.0],
                    [-0.5, 0.5, 0.0],
                    [-0.5, 0.0, 0.0],
                    [0.0, 0.0, 0.0],
                ],
            },
        },
    },
    "diaspore_alpha": {
        "db_key": ("diaspore_alpha.su.ifn", "phonons_setup_rel", 100),
        "structure": ROOT / "optimized_best/dft/diaspore_alpha.vasp",
        "dim": IDENTITY,
        "calc_ver": 2,  # force-relaxed α; avoid colliding with EOS disp_* v1
        "spacegroup": "Pnma",
        "paths": {
            "in_plane": {
                "labels": [r"$\Gamma$", "X", "S", "Y", r"$\Gamma$"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [0.5, 0.0, 0.0],
                    [0.5, 0.5, 0.0],
                    [0.0, 0.5, 0.0],
                    [0.0, 0.0, 0.0],
                ],
            },
            "out_of_plane": {
                "labels": [r"$\Gamma$", "Z", "U", "R", "T", "Z"],
                "qpoints": [
                    [0.0, 0.0, 0.0],
                    [0.0, 0.0, 0.5],
                    [0.5, 0.0, 0.5],
                    [0.5, 0.5, 0.5],
                    [0.0, 0.5, 0.5],
                    [0.0, 0.0, 0.5],
                ],
            },
        },
        "vertical": {
            "labels": ["X", "U", "|", "Y", "T", "|", "S", "R"],
            "qpoints": [
                [0.5, 0.0, 0.0],
                [0.5, 0.0, 0.5],
                None,
                [0.0, 0.5, 0.0],
                [0.0, 0.5, 0.5],
                None,
                [0.5, 0.5, 0.0],
                [0.5, 0.5, 0.5],
            ],
        },
    },
}


In [5]:
def phase_dir(phase):
    p = DISP_ROOT / phase
    p.mkdir(parents=True, exist_ok=True)
    return p


def job_name(phase, i):
    """Unique siman name so each displacement gets its own folder and sbatch."""
    return f"disp_{phase}_{i:03d}"


def it_folder_for(phase, i):
    return f"phonons_dispersion/{phase}/{i:03d}"


def phase_calc_ver(phase):
    return PHASES[phase].get("calc_ver", CALC_VER)


def db_key(phase, i):
    return (job_name(phase, i), VASP_SET, phase_calc_ver(phase))


def ase_to_phonopy(atoms):
    return PhonopyAtoms(
        symbols=atoms.get_chemical_symbols(),
        cell=atoms.get_cell().array,
        scaled_positions=atoms.get_scaled_positions(),
    )


def phonopy_to_ase(ph_atoms):
    return Atoms(
        symbols=ph_atoms.symbols,
        cell=ph_atoms.cell,
        scaled_positions=ph_atoms.scaled_positions,
        pbc=True,
    )


def write_relaxed_from_db(phase=None):
    """Write PHASES[*]['db_key'] .end supercells. Do not reduce or replicate."""
    phases = [phase] if phase else list(PHASES)
    written = []
    for name in phases:
        cfg = PHASES[name]
        key = cfg["db_key"]
        st = db[key].copy().end
        dest = Path(cfg["structure"])
        dest.parent.mkdir(parents=True, exist_ok=True)
        st.write_poscar(str(dest))
        lengths = np.linalg.norm(np.array(st.rprimd), axis=1)
        maxf = getattr(db[key], "maxforce", None)
        print(
            f"{name}: {len(st.xred)} atoms  abc={lengths} Å  "
            f"maxforce={maxf} meV/A  → {dest}"
        )
        if np.min(lengths) < 10.0:
            raise ValueError(f"{name}: a lattice vector is < 10 Å: {lengths}")
        written.append(dest)
    return written


def path_segments(phase, phase_cfg, include_vertical=True):
    """Return [(name, labels, qpoints), ...] in plot order."""
    order = {
        "layered_P2_1_c": ["out_of_plane", "in_plane"],
        "diaspore_alpha": ["in_plane", "out_of_plane"],
    }[phase]
    segs = []
    for key in order:
        p = phase_cfg["paths"][key]
        segs.append((key, p["labels"], np.array(p["qpoints"], dtype=float)))
    if include_vertical and "vertical" in phase_cfg:
        raw_l = phase_cfg["vertical"]["labels"]
        raw_q = phase_cfg["vertical"]["qpoints"]
        cur_l, cur_q = [], []
        n = 0
        for lab, q in zip(raw_l, raw_q):
            if q is None or lab == "|":
                if cur_q:
                    n += 1
                    segs.append((f"vertical_{n}", cur_l, np.array(cur_q, dtype=float)))
                cur_l, cur_q = [], []
                continue
            cur_l.append(lab)
            cur_q.append(q)
        if cur_q:
            n += 1
            segs.append((f"vertical_{n}", cur_l, np.array(cur_q, dtype=float)))
    return segs


def write_band_conf(phase, phase_cfg, dest):
    dim = phase_cfg["dim"]
    lines = [
        "ATOM_NAME = Ni H O F",
        f"DIM = {dim[0][0]} {dim[1][1]} {dim[2][2]}",
        "PRIMITIVE_AXES = AUTO",
        "BAND_POINTS = 51",
    ]
    born = phase_dir(phase) / "BORN"
    if born.exists():
        lines.append("NAC = .TRUE.")
    all_labels = []
    for name, labels, qs in path_segments(phase, phase_cfg):
        flat = "  ".join(f"{x:.6f} {y:.6f} {z:.6f}" for x, y, z in qs)
        lines.append(f"# {name}")
        lines.append(f"BAND = {flat}")
        if all_labels:
            all_labels.append("|")
        all_labels.extend(labels)
    token = []
    for lab in all_labels:
        token.append("G" if lab in (r"$\Gamma$", "G") else lab.replace("$", "").replace("\\", ""))
    lines.append("BAND_LABELS = " + " ".join(token))
    dest.write_text("\n".join(lines) + "\n")
    return dest


def nac_job_name(phase):
    return f"nac_{phase}"


def nac_it_folder(phase):
    return f"phonons_dispersion/{phase}/nac"


def nac_db_key(phase):
    return (nac_job_name(phase), NAC_SET, phase_calc_ver(phase))


def prepare_nac_poscar(phase):
    """Write phonopy primitive as POSCAR for LEPSILON (BORN atom order)."""
    work = phase_dir(phase)
    yaml = work / "phonopy_disp.yaml"
    if not yaml.exists():
        generate_displacements(phase)
    atoms = read(str(work / "POSCAR"))
    phonon = Phonopy(
        ase_to_phonopy(atoms),
        supercell_matrix=PHASES[phase]["dim"],
        primitive_matrix="auto",
    )
    nac_dir = work / "nac"
    nac_dir.mkdir(parents=True, exist_ok=True)
    write_vasp(str(nac_dir / "POSCAR"), phonon.primitive)
    write_vasp(str(work / "PPOSCAR"), phonon.primitive)
    print(f"{phase}: NAC primitive {len(phonon.primitive)} atoms → {nac_dir / 'POSCAR'}")
    return nac_dir / "POSCAR"


def parse_vasp_born_dielectric(outcar_path):
    """Parse ε∞ and Born effective charges from a VASP LEPSILON OUTCAR."""
    text = Path(outcar_path).read_text(errors="ignore")
    lines = text.splitlines()

    def _read_tensor_after(header_substr):
        for i, line in enumerate(lines):
            if header_substr in line:
                rows = []
                for j in range(i + 1, len(lines)):
                    parts = lines[j].split()
                    if len(parts) >= 3:
                        try:
                            rows.append([float(parts[-3]), float(parts[-2]), float(parts[-1])])
                        except ValueError:
                            if rows:
                                break
                            continue
                    if len(rows) == 3:
                        break
                if len(rows) == 3:
                    return np.array(rows, dtype=float)
        return None

    eps = _read_tensor_after("MACROSCOPIC STATIC DIELECTRIC TENSOR (including local field effects in DFT)")
    if eps is None:
        eps = _read_tensor_after("MACROSCOPIC STATIC DIELECTRIC TENSOR")
    if eps is None:
        raise RuntimeError(f"dielectric tensor not found in {outcar_path}")

    born = []
    capturing = False
    current = []
    for line in lines:
        if "BORN EFFECTIVE CHARGES" in line and "ionic" not in line.lower():
            capturing = True
            current = []
            continue
        if not capturing:
            continue
        if line.strip().startswith("ion "):
            if current:
                born.append(np.array(current, dtype=float))
                current = []
            continue
        parts = line.split()
        if len(parts) >= 4 and parts[0] in ("1", "2", "3"):
            try:
                current.append([float(parts[1]), float(parts[2]), float(parts[3])])
            except ValueError:
                pass
            if len(current) == 3:
                born.append(np.array(current, dtype=float))
                current = []
        # stop at next major block
        if capturing and born and (
            "polarization" in line.lower()
            or line.startswith("------")
            and "BORN" not in line
        ):
            # keep going until we leave the block; break on empty + already have charges
            pass
        if "PIEZOELECTRIC TENSOR" in line or "elasticity" in line.lower():
            break
    if current and len(current) == 3:
        born.append(np.array(current, dtype=float))

    if not born:
        raise RuntimeError(f"Born charges not found in {outcar_path}")
    return eps, born


def write_phonopy_born(dest, epsilon, born_charges, factor=14.399652):
    """Write phonopy BORN file (primitive-cell atom order)."""
    lines = [f"{factor:.6f}"]
    for row in epsilon:
        lines.append(f"{row[0]:16.8f} {row[1]:16.8f} {row[2]:16.8f}")
    for z in born_charges:
        for row in z:
            lines.append(f"{row[0]:16.8f} {row[1]:16.8f} {row[2]:16.8f}")
    Path(dest).write_text("\n".join(lines) + "\n")
    print(f"  wrote {dest}  (n_atoms_born={len(born_charges)})")
    return dest


write_relaxed_from_db()


-- POSCAR was written to /home/a.burov/icys_2025/niohf/optimized_best/dft/layered_P2_1_c.vasp 

layered_P2_1_c: 128 atoms  abc=[10.7404554  12.35814645 10.22651395] Å  maxforce=1 meV/A  → /home/a.burov/icys_2025/niohf/optimized_best/dft/layered_P2_1_c.vasp
-- POSCAR was written to /home/a.burov/icys_2025/niohf/optimized_best/dft/diaspore_alpha.vasp 

diaspore_alpha: 192 atoms  abc=[12.36737369 14.06580778 10.18253734] Å  maxforce=1 meV/A  → /home/a.burov/icys_2025/niohf/optimized_best/dft/diaspore_alpha.vasp


[PosixPath('/home/a.burov/icys_2025/niohf/optimized_best/dft/layered_P2_1_c.vasp'),
 PosixPath('/home/a.burov/icys_2025/niohf/optimized_best/dft/diaspore_alpha.vasp')]

In [6]:
def generate_displacements(phase, distance=None):
    if distance is None:
        distance = DISP_DISTANCE
    cfg = PHASES[phase]
    work = phase_dir(phase)
    write_relaxed_from_db(phase)
    atoms = read(str(cfg["structure"]))
    write(str(work / "POSCAR"), atoms, format="vasp", direct=True)

    # Cell is already the 2x2x2 / 4x3x1 supercell; do not seekpath-reduce it.
    phonon = Phonopy(
        ase_to_phonopy(atoms),
        supercell_matrix=cfg["dim"],
        primitive_matrix="auto",
    )
    phonon.generate_displacements(distance=distance)
    phonon.save(str(work / "phonopy_disp.yaml"), settings={"force_sets": False})

    sposcar = phonon.supercell
    write_vasp(str(work / "SPOSCAR"), sposcar)
    # Primitive used for NAC / BORN (atom order must match BORN file)
    write_vasp(str(work / "PPOSCAR"), phonon.primitive)
    disp_cells = phonon.supercells_with_displacements
    n_disp = 0
    for i, sc in enumerate(disp_cells, start=1):
        if sc is None:
            continue
        write_vasp(str(work / f"POSCAR-{i:03d}"), sc)
        n_disp += 1

    write_band_conf(phase, cfg, work / "band.conf")
    print(
        f"{phase}: {n_disp} displacements @ {distance} Å, "
        f"dim={cfg['dim'][0][0]}x{cfg['dim'][1][1]}x{cfg['dim'][2][2]}, "
        f"primitive atoms={len(phonon.primitive)}"
    )
    print(f"  work dir: {work}")
    return n_disp


def list_displacement_ids(phase):
    work = phase_dir(phase)
    ids = []
    for p in work.glob("POSCAR-*"):
        ids.append(int(p.name.split("-")[-1]))
    return sorted(ids)


## Generate displacements and launch with siman

Each `POSCAR-00N` is a **separate** siman job and folder:

`phonons_dispersion/{phase}/{N:03d}/disp_{phase}_{N:03d}.phonons_disp/`

zen4, `srun vasp_gam`, Gamma-only `ngkpt=[1,1,1]`, **`EDIFF=1e-8`**, **`LREAL=.FALSE.`**, displacement **`DISP_DISTANCE` (default 0.0075 Å)**, 24 h / 64 GB per displacement.


In [7]:
def launch_phase(phase, run=2):
    generate_displacements(phase)
    work = phase_dir(phase)
    ids = list_displacement_ids(phase)
    print(f"\nLaunching {phase}: {len(ids)} separate zen4 jobs  (run={run})")
    for i in ids:
        key = db_key(phase, i)
        if SKIP_EXISTING and key in db:
            print(f"  skip {i} (already in db as {key[0]})")
            continue
        poscar = work / f"POSCAR-{i:03d}"
        st = smart_structure_read(str(poscar))
        add(
            job_name(phase, i),
            VASP_SET,
            phase_calc_ver(phase),
            it_folder=it_folder_for(phase, i),
            input_st=st,
            ngkpt=[1, 1, 1],
            cluster=CLUSTER,
            up="up2",
            run=run,
        )
    return ids


if LAUNCH_JOBS:
    for phase in PHASES:
        launch_phase(phase, run=2)
    write_database()
    print("\nDatabase updated. Jobs submitted (or inputs written).")
else:
    for phase in PHASES:
        generate_displacements(phase)
    print("LAUNCH_JOBS=False — displacements only, no siman add()")


-- POSCAR was written to /home/a.burov/icys_2025/niohf/optimized_best/dft/layered_P2_1_c.vasp 

layered_P2_1_c: 128 atoms  abc=[10.7404554  12.35814645 10.22651395] Å  maxforce=1 meV/A  → /home/a.burov/icys_2025/niohf/optimized_best/dft/layered_P2_1_c.vasp
layered_P2_1_c: 24 displacements, dim=1x1x1
  work dir: /home/a.burov/icys_2025/niohf/phonons_dispersion/layered_P2_1_c

Launching layered_P2_1_c: 24 separate zen4 jobs  (run=2)
  skip 1 (already in db as disp_layered_P2_1_c_001)
  skip 2 (already in db as disp_layered_P2_1_c_002)
  skip 3 (already in db as disp_layered_P2_1_c_003)
  skip 4 (already in db as disp_layered_P2_1_c_004)
  skip 5 (already in db as disp_layered_P2_1_c_005)
  skip 6 (already in db as disp_layered_P2_1_c_006)
  skip 7 (already in db as disp_layered_P2_1_c_007)
  skip 8 (already in db as disp_layered_P2_1_c_008)
  skip 9 (already in db as disp_layered_P2_1_c_009)
  skip 10 (already in db as disp_layered_P2_1_c_010)
  skip 11 (already in db as disp_layered_P2_

## NAC (Born charges + ε∞) and LO–TO splitting

Polar crystals need the **non-analytic correction** for correct LO–TO splitting near Γ.

1. `prepare_nac_poscar` writes the **phonopy primitive** (`PPOSCAR` / `nac/POSCAR`)
2. `LAUNCH_NAC=True` submits `phonons_nac` (`LEPSILON=.TRUE.`, `EDIFF=1e-8`, `LREAL=.FALSE.`, dense `KSPACING`)
3. After the job finishes, `build_born_file` parses OUTCAR → `BORN`
4. Band structure / LO–TO plots use NAC when `USE_NAC=True` and `BORN` exists

Force jobs stay Γ-only; NAC is a **separate** denser-k primitive calculation.


In [ ]:
def _outcar_finished(outcar: Path) -> bool:
    if not outcar.exists() or "unfinished" in outcar.name:
        return False
    try:
        tail = outcar.read_text(errors="ignore")[-8000:]
    except Exception:
        return outcar.stat().st_size > 50000
    return ("General timing" in tail) or ("Voluntary number of scf" in tail)


def launch_nac(phase, run=2):
    """Submit DFPT LEPSILON job on the phonopy primitive cell."""
    poscar = prepare_nac_poscar(phase)
    key = nac_db_key(phase)
    if SKIP_EXISTING and key in db:
        print(f"  skip NAC {phase} (already in db)")
        return key
    st = smart_structure_read(str(poscar))
    # denser mesh via set KSPACING; do not force ngkpt=[1,1,1]
    add(
        nac_job_name(phase),
        NAC_SET,
        phase_calc_ver(phase),
        it_folder=nac_it_folder(phase),
        input_st=st,
        cluster=CLUSTER,
        up="up2",
        run=run,
    )
    print(f"  submitted {key}")
    return key


def nac_outcar_path(phase):
    key = nac_db_key(phase)
    candidates = []
    if key in db:
        calc = db[key]
        path = calc.path.get("output") if hasattr(calc, "path") else None
        if path:
            candidates.append(Path(path))
            candidates.append(ROOT / path)
            candidates.append(Path(str(path).replace("OUTCAR_unfinished", "OUTCAR")))
    folder = ROOT / nac_it_folder(phase)
    candidates.extend(folder.glob("**/1.OUTCAR"))
    candidates.extend(folder.glob("**/*.OUTCAR"))
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if p.is_file() and _outcar_finished(p):
            return p
    return None


def build_born_file(phase):
    """res() NAC job if needed, parse OUTCAR, write phonopy BORN in phase dir."""
    work = phase_dir(phase)
    try:
        res(nac_job_name(phase), NAC_SET, phase_calc_ver(phase), up="up2", cluster=CLUSTER)
    except Exception as exc:
        print(f"  res() NAC {phase}: {exc}")
    oc = nac_outcar_path(phase)
    if oc is None:
        print(f"{phase}: NAC OUTCAR not finished yet")
        return None
    eps, born = parse_vasp_born_dielectric(oc)
    print(f"{phase}: ε∞ =\n{eps}")
    print(f"{phase}: Born charges for {len(born)} primitive atoms")
    dest = work / "BORN"
    write_phonopy_born(dest, eps, born)
    write_band_conf(phase, PHASES[phase], work / "band.conf")
    return dest


if LAUNCH_NAC:
    for phase in PHASES:
        launch_nac(phase, run=2)
    write_database()
    print("NAC jobs submitted.")
else:
    print("LAUNCH_NAC=False — set True to submit LEPSILON Born/ε∞ jobs")


## Read results and build dispersions

Run this cell after the queue finishes. Incomplete displacements are reported and skipped.


In [ ]:
import sys
from pathlib import Path

def _repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "plot_colors.py").is_file():
            return cand
    raise FileNotFoundError("plot_colors.py not found from " + str(here))

sys.path.insert(0, str(_repo_root()))
from plot_colors import series_colors

def _looks_finished(outcar: Path) -> bool:
    if not outcar.exists() or "unfinished" in outcar.name:
        return False
    try:
        tail = outcar.read_text(errors="ignore")[-8000:]
    except Exception:
        return outcar.stat().st_size > 50000
    return ("General timing" in tail) or ("Voluntary number of scf" in tail)


def outcar_path(phase, i):
    """Finished OUTCAR for displacement i (new per-folder layout or db path)."""
    candidates = []
    key = db_key(phase, i)
    if key in db:
        calc = db[key]
        path = calc.path.get("output") if hasattr(calc, "path") else None
        if path:
            candidates.append(Path(path))
            candidates.append(ROOT / path)
            candidates.append(Path(str(path).replace("OUTCAR_unfinished", "OUTCAR")))

    folder = ROOT / it_folder_for(phase, i)
    candidates.extend(folder.glob("**/1.OUTCAR"))
    candidates.extend(folder.glob("**/*.OUTCAR"))

    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if p in seen or not p.is_file():
            continue
        seen.add(p)
        if _looks_finished(p):
            return p
    return None


def read_phase(phase):
    ids = list_displacement_ids(phase)
    done, missing = [], []
    for i in ids:
        try:
            res(job_name(phase, i), VASP_SET, phase_calc_ver(phase), up="up2", cluster=CLUSTER)
        except Exception as exc:
            print(f"  res() {phase} {i}: {exc}")
        if outcar_path(phase, i) is not None:
            done.append(i)
        else:
            missing.append(i)
    print(f"{phase}: {len(done)} finished, {len(missing)} missing {missing[:12]}")
    return done, missing


def collect_force_files(phase, done_ids):
    """Copy finished OUTCAR/vasprun into the phase dir as 001.OUTCAR for phonopy -f."""
    work = phase_dir(phase)
    files = []
    for i in done_ids:
        oc = outcar_path(phase, i)
        if oc is None:
            continue
        dest = work / f"{i:03d}.OUTCAR"
        if oc.resolve() != dest.resolve():
            shutil.copy2(oc, dest)
        for cand in (
            oc.with_name(oc.name.replace("OUTCAR", "vasprun.xml")),
            oc.parent / "vasprun.xml",
            oc.parent / "1.vasprun.xml",
        ):
            if cand.exists() and "unfinished" not in cand.name:
                vdest = work / f"{i:03d}.vasprun.xml"
                if cand.resolve() != vdest.resolve():
                    shutil.copy2(cand, vdest)
                files.append(vdest)
                break
        else:
            files.append(dest)
    return files


def run_phonopy_forces(phase, force_files):
    work = phase_dir(phase)
    yaml = work / "phonopy_disp.yaml"
    if not yaml.exists():
        raise FileNotFoundError(f"missing {yaml}; regenerate displacements first")

    phonopy_bin = shutil.which("phonopy")
    if phonopy_bin:
        cmd = [phonopy_bin, "-f"] + [str(p.name) for p in force_files]
        result = subprocess.run(cmd, cwd=work, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"phonopy -f failed for {phase}")
        return work / "FORCE_SETS"

    n_atoms = len(read(str(work / "SPOSCAR")))
    forces = parse_set_of_forces(n_atoms, [str(p) for p in force_files])
    write_FORCE_SETS(forces, filename=str(work / "FORCE_SETS"))
    return work / "FORCE_SETS"


def build_band_path(phase, phase_cfg, npoints=51):
    bands, labels, path_connections = [], [], []
    for _, seg_labels, qs in path_segments(phase, phase_cfg):
        nseg = len(qs) - 1
        for j in range(nseg):
            bands.append(np.linspace(qs[j], qs[j + 1], npoints))
        path_connections.extend([True] * (nseg - 1) + [False])
        labels.extend(seg_labels)
    return bands, labels, path_connections


def _band_xticks(distances, path_connections, labels):
    xticks, xlabels = [], []
    lab_i = 0
    for i, dist in enumerate(distances):
        new_group = i == 0 or not path_connections[i - 1]
        if new_group:
            xticks.append(float(dist[0]))
            xlabels.append(labels[lab_i] if lab_i < len(labels) else "")
            lab_i += 1
        xticks.append(float(dist[-1]))
        xlabels.append(labels[lab_i] if lab_i < len(labels) else "")
        lab_i += 1
    return xticks, xlabels


def _freq_break(frequencies, gap_min=8.0):
    """Find a large empty gap between lattice modes and O–H stretches."""
    vals = np.concatenate([np.asarray(f).ravel() for f in frequencies])
    vals = vals[np.isfinite(vals)]
    if vals.size < 10:
        return None
    lo, hi = float(np.min(vals)), float(np.max(vals))
    # ignore deep imaginary modes for gap search
    pos = np.sort(vals[vals > -5.0])
    if pos.size < 10:
        return None
    d = np.diff(pos)
    i = int(np.argmax(d))
    if d[i] < gap_min:
        return None
    mid_lo = float(pos[i])
    mid_hi = float(pos[i + 1])
    # require the upper block to be a thin stretch band near the top
    if mid_hi - mid_lo < gap_min:
        return None
    return (lo, mid_lo, mid_hi, hi)


def _style_band_ax(ax, xticks, xlabels, ylabel=None, show_x=True):
    ax.axhline(0.0, color="#9aa3ad", lw=0.7, ls="--", zorder=0)
    for x in xticks:
        ax.axvline(x, color="#d9dee5", lw=0.6, zorder=0)
    ax.set_xlim(xticks[0], xticks[-1])
    ax.tick_params(axis="both", labelsize=11, length=3.5, width=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=12)
    if show_x:
        ax.set_xticks(xticks)
        ax.set_xticklabels(xlabels, fontsize=12)
    else:
        ax.set_xticks(xticks)
        ax.set_xticklabels([])


def plot_and_save_bands(phase, phonon):
    """Publication-style bands with an optional broken y-axis across the O–H gap."""
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    plt.close("all")
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
        "axes.linewidth": 1.0,
        "figure.dpi": 120,
    })

    cfg = PHASES[phase]
    bands, labels, path_connections = build_band_path(phase, cfg)
    color = "#1f4e79"
    lw = 0.95

    try:
        data = phonon.get_band_structure_dict()
        distances = data["distances"]
        frequencies = data["frequencies"]
    except Exception as exc:
        print(f"  custom extract failed ({exc}); phonopy default plot")
        phonon.plot_band_structure()
        fig = plt.gcf()
        fig.set_size_inches(10, 5.2)
        out_png = FIGURES_ROOT / f"dft_{phase}_bands.png"
        out_pdf = FIGURES_ROOT / f"dft_{phase}_bands.pdf"
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        fig.savefig(out_pdf, bbox_inches="tight")
        plt.close(fig)
        print(f"  saved {out_png}")
        return

    xticks, xlabels = _band_xticks(distances, path_connections, labels)
    brk = _freq_break(frequencies)

    if brk is None:
        fig, ax = plt.subplots(figsize=(10.5, 5.4))
        for dist, freq in zip(distances, frequencies):
            ax.plot(dist, freq, color=color, lw=lw)
        all_f = np.concatenate([np.asarray(f).ravel() for f in frequencies])
        ymin = min(-2.0, float(np.nanmin(all_f)) - 1.0)
        ymax = float(np.nanmax(all_f)) * 1.04
        ax.set_ylim(ymin, ymax)
        _style_band_ax(ax, xticks, xlabels, ylabel="Frequency (THz)")
        ax.set_title(f"DFT — {phase}", fontsize=13, pad=8)
    else:
        lo, mid_lo, mid_hi, hi = brk
        # pad the two visible windows
        y_lo_max = mid_lo + 0.08 * max(mid_lo - lo, 5.0)
        y_hi_min = mid_hi - 0.08 * max(hi - mid_hi, 2.0)
        y_hi_max = hi + 0.04 * max(hi - mid_hi, 2.0)
        # keep a little room for soft imaginary modes
        y_lo_min = min(-2.0, lo - 1.0)

        fig, (ax_hi, ax_lo) = plt.subplots(
            2,
            1,
            sharex=True,
            figsize=(10.5, 6.2),
            gridspec_kw={"height_ratios": [1.0, 3.2], "hspace": 0.05},
        )
        for ax in (ax_hi, ax_lo):
            for dist, freq in zip(distances, frequencies):
                ax.plot(dist, freq, color=color, lw=lw)

        ax_hi.set_ylim(y_hi_min, y_hi_max)
        ax_lo.set_ylim(y_lo_min, y_lo_max)
        _style_band_ax(ax_hi, xticks, xlabels, show_x=False)
        _style_band_ax(ax_lo, xticks, xlabels, ylabel="Frequency (THz)")
        ax_hi.set_title(f"DFT — {phase}", fontsize=13, pad=8)

        # diagonal break marks
        d = 0.012
        kwargs = dict(transform=ax_hi.transAxes, color="0.35", clip_on=False, lw=0.9)
        ax_hi.plot((-d, +d), (-d, +d), **kwargs)
        ax_hi.plot((1 - d, 1 + d), (-d, +d), **kwargs)
        kwargs.update(transform=ax_lo.transAxes)
        ax_lo.plot((-d, +d), (1 - d, 1 + d), **kwargs)
        ax_lo.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
        ax_hi.spines["bottom"].set_visible(False)
        ax_lo.spines["top"].set_visible(False)
        ax_hi.tick_params(bottom=False)

    out_png = FIGURES_ROOT / f"dft_{phase}_bands.png"
    out_pdf = FIGURES_ROOT / f"dft_{phase}_bands.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out_png}" + ("  [broken y-axis]" if brk else ""))


def apply_nac(phonon, work):
    """Attach phonopy nac_params from BORN if USE_NAC and file exists."""
    born = Path(work) / "BORN"
    if not USE_NAC:
        print("  NAC disabled (USE_NAC=False)")
        return phonon
    if not born.exists():
        print(f"  no {born} — bands without NAC / LO–TO")
        return phonon
    try:
        from phonopy.file_IO import parse_BORN
        nac = parse_BORN(phonon.primitive, filename=str(born))
        phonon.nac_params = nac
        print(f"  NAC loaded from {born}")
    except Exception as exc:
        # manual parse fallback
        try:
            lines = born.read_text().strip().splitlines()
            factor = float(lines[0].split()[0])
            eps = np.array([[float(x) for x in lines[i].split()[:3]] for i in range(1, 4)])
            born_list = []
            vals = []
            for line in lines[4:]:
                vals.extend([float(x) for x in line.split()[:3]])
                while len(vals) >= 9:
                    z = np.array(vals[:9]).reshape(3, 3)
                    born_list.append(z)
                    vals = vals[9:]
            phonon.nac_params = {
                "born": np.array(born_list),
                "dielectric": eps,
                "factor": factor,
            }
            print(f"  NAC loaded manually from {born}")
        except Exception as exc2:
            print(f"  NAC parse failed: {exc} / {exc2}")
    return phonon


def load_phonon(work):
    """Load FORCE_SETS + phonopy_disp.yaml; apply NAC when BORN is present."""
    work = Path(work)
    yaml = str(work / "phonopy_disp.yaml")
    fsets = str(work / "FORCE_SETS")
    born = work / "BORN"
    load_fn = phonopy_load or getattr(Phonopy, "load", None)
    if load_fn is not None:
        kw_try = []
        base = {"phonopy_yaml": yaml}
        for fs_key in ("force_sets_filename", "force_sets_file_name"):
            kw = dict(base)
            kw[fs_key] = fsets
            if USE_NAC and born.exists():
                kw["born_filename"] = str(born)
            kw_try.append(kw)
            kw_try.append({k: v for k, v in kw.items() if k != "born_filename"})
        for kwargs in kw_try:
            try:
                phonon = load_fn(**kwargs)
                return apply_nac(phonon, work)
            except TypeError:
                continue
            except Exception as exc:
                print(f"  load_fn failed ({exc}); trying next signature")

    from phonopy.interface.phonopy_yaml import PhonopyYaml

    phy = PhonopyYaml()
    phy.read(yaml)
    phonon = Phonopy(
        phy.unitcell,
        supercell_matrix=phy.supercell_matrix,
        primitive_matrix=phy.primitive_matrix,
    )
    if getattr(phy, "dataset", None) is not None:
        phonon.dataset = phy.dataset
    force_sets = parse_FORCE_SETS(filename=fsets)
    if hasattr(phonon, "set_displacement_dataset") and isinstance(force_sets, dict):
        phonon.set_displacement_dataset(force_sets)
    elif hasattr(phonon, "set_forces"):
        first = force_sets["first_atoms"] if isinstance(force_sets, dict) else force_sets
        phonon.set_forces([d["forces"] for d in first])
    else:
        phonon.dataset = force_sets
    if hasattr(phonon, "produce_force_constants"):
        phonon.produce_force_constants()
    else:
        phonon.get_force_constants()
    return apply_nac(phonon, work)


def run_bands(phonon, bands, labels, path_connections):
    kwargs = dict(
        is_band_connection=True,
        path_connections=path_connections,
        labels=labels,
    )
    # with_nac is default when nac_params set; some versions expose the flag
    for extra in ({"with_nac": True}, {}):
        try:
            phonon.run_band_structure(bands, **kwargs, **extra)
            return
        except (TypeError, AttributeError):
            continue
    try:
        phonon.set_band_structure(bands, is_band_connection=True, labels=labels)
    except TypeError:
        phonon.set_band_structure(bands)


def plot_loto_splitting(phase, qmax=0.05, npoints=41):
    """
    LO–TO splitting: frequencies approaching Γ along x, y, z.
    Requires FORCE_SETS + BORN (NAC).
    """
    work = phase_dir(phase)
    if USE_NAC and not (work / "BORN").exists():
        build_born_file(phase)
    if not (work / "FORCE_SETS").exists():
        print(f"{phase}: need FORCE_SETS for LO–TO plot")
        return
    phonon = load_phonon(work)
    if getattr(phonon, "nac_params", None) is None:
        print(f"{phase}: no nac_params — LO–TO plot skipped")
        return

    directions = {
        r"$q \parallel a^{*}$": np.array([1.0, 0.0, 0.0]),
        r"$q \parallel b^{*}$": np.array([0.0, 1.0, 0.0]),
        r"$q \parallel c^{*}$": np.array([0.0, 0.0, 1.0]),
    }
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    colors = series_colors(len(directions))
    for (lab, vec), color in zip(directions.items(), colors):
        v = vec / np.linalg.norm(vec)
        qs = np.linspace(qmax, 0.0, npoints)[:, None] * v[None, :]
        bands = [qs]
        try:
            phonon.run_band_structure(bands, with_nac=True)
        except TypeError:
            phonon.run_band_structure(bands)
        data = phonon.get_band_structure_dict()
        dist = np.asarray(data["distances"][0])
        # map distance so Γ is at 0 on the right
        x = dist[-1] - dist
        freq = np.asarray(data["frequencies"][0])
        ax.plot(x, freq, color=color, lw=0.9, label=lab)

    ax.axhline(0.0, color="#9aa3ad", lw=0.7, ls="--")
    ax.set_xlabel(r"Wave vector toward $\Gamma$ (reduced)", fontsize=12)
    ax.set_ylabel("Frequency (THz)", fontsize=12)
    ax.set_title(f"DFT LO–TO splitting — {phase}", fontsize=13)
    ax.legend(frameon=False, fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
    out_png = FIGURES_ROOT / f"dft_{phase}_loto.png"
    out_pdf = FIGURES_ROOT / f"dft_{phase}_loto.pdf"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved {out_png}")


def process_phase(phase, done_ids):
    cfg = PHASES[phase]
    work = phase_dir(phase)
    force_files = collect_force_files(phase, done_ids)
    if len(force_files) < 3:
        print(f"{phase}: only {len(force_files)} force files — skip phonopy")
        return False

    expected = list_displacement_ids(phase)
    if expected and len(force_files) != len(expected):
        print(
            f"{phase}: {len(force_files)}/{len(expected)} forces; "
            "FORCE_SETS needs the full set in order — skip until complete"
        )
        return False

    print(f"\n=== phonopy {phase} ({len(force_files)} files) ===")
    run_phonopy_forces(phase, force_files)

    if USE_NAC:
        if not (work / "BORN").exists():
            build_born_file(phase)

    phonon = load_phonon(work)
    bands, labels, path_connections = build_band_path(phase, cfg)
    run_bands(phonon, bands, labels, path_connections)
    try:
        phonon.write_yaml_band_structure(filename=str(work / "band.yaml"))
    except TypeError:
        phonon.write_yaml_band_structure()
        src = Path("band.yaml")
        if src.exists():
            src.replace(work / "band.yaml")
    plot_and_save_bands(phase, phonon)
    try:
        plot_loto_splitting(phase)
    except Exception as exc:
        print(f"  LO–TO plot failed: {exc}")
    return True


In [9]:
if READ_AND_PROCESS:
    results = {}
    for phase in PHASES:
        # regenerate yaml/POSCAR/band.conf if this tree was copied without them
        if not (phase_dir(phase) / "phonopy_disp.yaml").exists():
            generate_displacements(phase)
        done, missing = read_phase(phase)
        results[phase] = {"done": done, "missing": missing}
        if missing:
            print(f"  wait for jobs disp_{phase}_XXX / {VASP_SET} / {missing}")
            continue
        process_phase(phase, done)
    write_database()
else:
    print("READ_AND_PROCESS=False")


The following TM are found: [8, 28]

 Znucl:   8
  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09

 Znucl:   28
  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84  1.84


--                           |  energy(eV)|    Vector lenghts (A)   | Stresses (MPa)     | N MD, N SCF    


Max. F. tot  (meV/A) =  [29] 

-- db['disp_layered_P2_1_c_001.phonons_disp.1']| -621.9832  |10.74, 12.36, 10.23|  -443,-449,-471    |   1,37, 37    
The following TM are found: [8, 28]

 Znucl:   8
  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09  0.09

 Znucl:   28
  1.84  1.84  1.84  1.84  1.84  1.8

## Imaginary (negative) frequencies

Lists soft modes along the plotted band path: **q** (fractional, primitive BZ), frequency in **THz** and **eV**  
(\(E = h\nu\), \(h = 4.135667696\times10^{-3}\) eV/THz).


In [ ]:
# E = h * f ; phonopy band frequencies are in THz by default
THZ_TO_EV = 4.135667696e-3  # eV / THz


def _q_at_distance(distances, qpoints, d_target):
    """Interpolate fractional q along a band segment for a given path distance."""
    d = np.asarray(distances, dtype=float)
    q = np.asarray(qpoints, dtype=float)
    if d.size < 2:
        return q[0]
    # phonopy distances are cumulative along the segment
    return np.array([
        np.interp(d_target, d, q[:, i]) for i in range(3)
    ])


def report_imaginary_modes(phase, thresh_thz=1e-4, top_n=30):
    """
    Print negative phonon frequencies along the current band path.

    Needs FORCE_SETS + phonopy_disp.yaml (same as process_phase).
    """
    work = phase_dir(phase)
    cfg = PHASES[phase]
    if not (work / "FORCE_SETS").exists() or not (work / "phonopy_disp.yaml").exists():
        print(f"{phase}: missing FORCE_SETS / phonopy_disp.yaml under {work}")
        return None

    phonon = load_phonon(work)
    bands, labels, path_connections = build_band_path(phase, cfg)
    run_bands(phonon, bands, labels, path_connections)
    data = phonon.get_band_structure_dict()

    rows = []
    # reconstruct label context per segment
    lab_i = 0
    for seg_i, (dist, freq, qs) in enumerate(
        zip(data["distances"], data["frequencies"], bands)
    ):
        new_group = seg_i == 0 or not path_connections[seg_i - 1]
        if new_group:
            start_lab = labels[lab_i] if lab_i < len(labels) else "?"
            lab_i += 1
        end_lab = labels[lab_i] if lab_i < len(labels) else "?"
        lab_i += 1
        seg_name = f"{start_lab} → {end_lab}"

        dist = np.asarray(dist, dtype=float)
        freq = np.asarray(freq, dtype=float)  # (nq, nband)
        qs = np.asarray(qs, dtype=float)
        for iq in range(freq.shape[0]):
            q = qs[iq] if qs.ndim == 2 else _q_at_distance(dist, qs, dist[iq])
            for ib, f in enumerate(freq[iq]):
                if f < -thresh_thz:
                    rows.append({
                        "phase": phase,
                        "segment": seg_name,
                        "q_frac": q,
                        "branch": ib,
                        "freq_THz": float(f),
                        "freq_eV": float(f) * THZ_TO_EV,
                        "freq_meV": float(f) * THZ_TO_EV * 1000.0,
                    })

    if not rows:
        print(f"{phase}: no imaginary modes below -{thresh_thz} THz on the plotted path")
        return []

    rows.sort(key=lambda r: r["freq_THz"])
    print(f"\n{phase}: {len(rows)} imaginary (q, branch) points  "
          f"(showing {min(top_n, len(rows))} most negative)\n")
    print(f"{'segment':22s} {'qx':>8s} {'qy':>8s} {'qz':>8s} "
          f"{'br':>4s} {'THz':>10s} {'meV':>10s} {'eV':>12s}")
    for r in rows[:top_n]:
        qx, qy, qz = r["q_frac"]
        print(
            f"{r['segment']:22s} {qx:8.4f} {qy:8.4f} {qz:8.4f} "
            f"{r['branch']:4d} {r['freq_THz']:10.4f} {r['freq_meV']:10.4f} {r['freq_eV']:12.6e}"
        )

    # unique softest mode summary
    softest = rows[0]
    print(
        f"\nSoftest: {softest['freq_THz']:.4f} THz = "
        f"{softest['freq_meV']:.4f} meV = {softest['freq_eV']:.6e} eV\n"
        f"  q = ({softest['q_frac'][0]:.4f}, {softest['q_frac'][1]:.4f}, "
        f"{softest['q_frac'][2]:.4f}) on {softest['segment']}, branch {softest['branch']}"
    )
    return rows


# Run for both phases when data is present
imag_all = {}
for phase in PHASES:
    imag_all[phase] = report_imaginary_modes(phase)


## Notes

- Structures: `db["layered_P2_1_c.sc", "phonons_setup_eos", 100].end` (P2₁/c, 2×2×2) and `db["diaspore_alpha.su.ifn", "phonons_setup_rel", 100].end` (Pnma, 4×3×1, force-relaxed)
- Force jobs (`phonons_disp`): `EDIFF=1e-8`, `LREAL=.FALSE.`, `DISP_DISTANCE=0.0075` Å
- **NAC / LO–TO**: set `LAUNCH_NAC=True` → `phonons_nac` (`LEPSILON`) on phonopy primitive → `BORN`; bands and `dft_*_loto` plots use it when `USE_NAC=True`
- Band figures: `figures/dispersion/dft_{phase}_bands.{png,pdf}` and `dft_{phase}_loto.{png,pdf}`
- No extra supercell: all lattice vectors already > 10 Å, `DIM = 1 1 1`, `primitive_matrix='auto'`
- Force jobs are Γ-only (`ngkpt=[1,1,1]`); NAC uses denser `KSPACING=0.35` on the **primitive**
- Set `LAUNCH_JOBS=False` / `LAUNCH_NAC=False` after submit; use `READ_AND_PROCESS` then `build_born_file` / replot
- Changing displacement amplitude requires **new** force jobs (do not mix older OUTCARs with a new `phonopy_disp.yaml`)
